# Isla-SNN v3 Inference 🧠⚡

Load a trained v3 checkpoint and generate text with your spiking model.

**v3 architecture:** 768d × 12L × 12H × 8 timesteps (~150M params)  
Gated residual + membrane features + spike frequency adaptation.

In [ ]:
!pip install -q torch transformers

## 1. Load Model

In [ ]:
import isla
import torch

CHECKPOINT_DIR = "./outputs/v3_150m/final"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = isla.load_model(CHECKPOINT_DIR, device=DEVICE)
print(f"Loaded: {model.count_params():,} params on {DEVICE}")
print(f"Config: {model.config.num_layers}L / {model.config.hidden_dim}D / {model.config.num_heads}H / {model.config.num_timesteps}T")

## 2. Generate Text

In [ ]:
text = isla.generate(
    model, tokenizer,
    prompt="The future of artificial intelligence",
    max_new_tokens=100,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.1,
    device=DEVICE,
)
print(text)

## 3. Streaming Generation

In [ ]:
for piece in isla.generate_stream(
    model, tokenizer,
    prompt="Once upon a time",
    max_new_tokens=100,
    temperature=0.7,
    device=DEVICE,
):
    print(piece, end="", flush=True)

## 4. Trilingual Prompts

v3 is trained on English, Portuguese, Code, and Math.

In [ ]:
prompts = [
    "The key difference between spiking and traditional neural networks",
    "A inteligência artificial está transformando",
    "def quicksort(arr):",
    "Question: If a train travels at 120 km/h for 2.5 hours, how far does it go?\nAnswer:",
]

for prompt in prompts:
    text = isla.generate(model, tokenizer, prompt,
                        max_new_tokens=80, temperature=0.7, device=DEVICE)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {text}\n")

## 5. Spike Rate Analysis

Check how sparse the model fires (lower = more efficient).

In [ ]:
model.eval()
input_ids = tokenizer.encode("Hello world", return_tensors="pt").to(DEVICE)

with torch.no_grad():
    logits, metrics, _ = model(input_ids)

print(f"Mean spike rate: {metrics['mean_spike_rate']:.4f}")
print(f"Spike rate std:  {metrics['spike_rate_std']:.4f}")
print()
for i, rate in enumerate(metrics['spike_rates_per_layer']):
    print(f"  Layer {i:2d}: {rate.mean():.4f}")

## 6. v3 Diagnostics

Inspect the learned gate values, membrane mixing (α), and SFA adaptation.

In [ ]:
print("=" * 50)
print("v3 Learned Parameters")
print("=" * 50)

for i, block in enumerate(model.blocks):
    gate = block.gate.item()
    alpha = block.mlp.alpha.item()
    beta = block.mlp.lif.beta.mean().item()
    sfa = torch.relu(block.mlp.lif.adaptation_strength).mean().item()
    tau = block.attn.tau.item() if hasattr(block.attn, 'tau') else 'N/A'
    
    print(f"  Layer {i:2d} | gate={gate:.3f} | α={alpha:.3f} | β={beta:.3f} | SFA={sfa:.3f} | τ={tau}")

## 7. Model Config

In [ ]:
import json
from pathlib import Path

config_path = Path(CHECKPOINT_DIR) / "model_config.json"
with open(config_path) as f:
    config = json.load(f)

print("Model Config:")
for k, v in config.items():
    print(f"  {k}: {v}")